# Part 2: TD Baseline Inference - Pre-trained TATR Performance

This notebook covers:
- Load pre-trained `microsoft/table-transformer-detection` model
- Run inference on test split images
- Compute evaluation metrics: IoU, mAP@50, mAP@50-95, F1-Score
- Visualize sample detections with bounding boxes

**Note:** No fine-tuning of the TD model is performed (per requirements).

**Reference:** Tablecert paper for evaluation protocol

## 2.1 Load Configuration from Part 1

In [ ]:
import os
import json
import pickle
import numpy as np
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoModelForObjectDetection,
    DetrImageProcessor,
    TableTransformerForObjectDetection,
)

from PIL import Image
import albumentations as A

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import pandas as pd
from tqdm.auto import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load configuration from Part 1 or define inline
from dataclasses import dataclass, field

@dataclass
class Config:
    """Global configuration for TATR TD and TSR."""
    IMAGES_DIR: str = "/kaggle/input/datasets/mohammedahmedxx12/machathon-dataset/Phase1_Train_Dataset/images"
    TD_ANNOTATIONS_DIR: str = "/kaggle/input/datasets/philopaterg/stp26-preprocessed-dataset/preprocessed_dataset/table_detection"
    TSR_ANNOTATIONS_DIR: str = "/kaggle/input/datasets/philopaterg/stp26-preprocessed-dataset/preprocessed_dataset/table_structure"
    OUTPUT_DIR: str = "/kaggle/working/outputs"
    TD_MODEL_NAME: str = "microsoft/table-transformer-detection"
    TSR_MODEL_NAME: str = "microsoft/table-transformer-structure-recognition"
    IMAGE_SIZE: int = 800
    BATCH_SIZE: int = 4
    SEED: int = 42
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
    TD_CATEGORIES: List[str] = field(default_factory=lambda: ["table"])

config = Config()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

# Set seed
import random
random.seed(config.SEED)
np.random.seed(config.SEED)
torch.manual_seed(config.SEED)

print(f"Configuration loaded: Device = {config.DEVICE}")

## 2.2 Load Pre-trained TD Model

In [ ]:
# Load pre-trained Table Detection model
print(f"Loading pre-trained model: {config.TD_MODEL_NAME}")

td_processor = DetrImageProcessor.from_pretrained(
    config.TD_MODEL_NAME,
    size={"shortest_edge": config.IMAGE_SIZE, "longest_edge": config.IMAGE_SIZE},
    do_resize=True,
    do_normalize=True,
)

td_model = TableTransformerForObjectDetection.from_pretrained(config.TD_MODEL_NAME)
td_model = td_model.to(config.DEVICE)
td_model.eval()

print(f"Model loaded successfully")
print(f"Model config: {td_model.config.num_labels} classes")
print(f"ID2Label: {td_model.config.id2label}")

## 2.3 Load Test Dataset

In [ ]:
class SimpleCOCODataset(Dataset):
    """Simple COCO dataset for inference."""
    
    def __init__(self, annotations_file: str, images_dir: str, processor: DetrImageProcessor):
        self.images_dir = images_dir
        self.processor = processor
        
        with open(annotations_file, 'r') as f:
            self.coco_data = json.load(f)
        
        self.images = {img['id']: img for img in self.coco_data['images']}
        self.categories = {cat['id']: cat['name'] for cat in self.coco_data['categories']}
        
        self.img_to_anns = defaultdict(list)
        for ann in self.coco_data['annotations']:
            self.img_to_anns[ann['image_id']].append(ann)
        
        self.image_ids = list(self.images.keys())
        print(f"Loaded {len(self.image_ids)} images")
    
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_info = self.images[image_id]
        
        image_path = os.path.join(self.images_dir, image_info['file_name'])
        image = Image.open(image_path).convert('RGB')
        orig_size = image.size  # (width, height)
        
        encoding = self.processor(images=image, return_tensors='pt')
        pixel_values = encoding['pixel_values'].squeeze(0)
        
        # Get ground truth
        gt_boxes = []
        gt_labels = []
        for ann in self.img_to_anns[image_id]:
            gt_boxes.append(ann['bbox'])  # [x, y, w, h]
            gt_labels.append(ann['category_id'])
        
        return {
            'pixel_values': pixel_values,
            'image_id': image_id,
            'orig_size': orig_size,
            'gt_boxes': gt_boxes,
            'gt_labels': gt_labels,
            'file_name': image_info['file_name'],
        }


# Load test dataset
test_file = os.path.join(config.TD_ANNOTATIONS_DIR, "test.json")
print(f"\nLoading test dataset from: {test_file}")

test_dataset = SimpleCOCODataset(
    annotations_file=test_file,
    images_dir=config.IMAGES_DIR,
    processor=td_processor
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,  # Process one image at a time for simplicity
    shuffle=False,
    num_workers=0
)

## 2.4 Evaluation Metrics Implementation

In [ ]:
def compute_iou(box1: List[float], box2: List[float]) -> float:
    """
    Compute IoU between two boxes in COCO format [x, y, w, h].
    
    Args:
        box1, box2: Boxes in [x, y, width, height] format
    
    Returns:
        IoU score
    """
    # Convert to [x1, y1, x2, y2]
    x1_1, y1_1 = box1[0], box1[1]
    x2_1, y2_1 = box1[0] + box1[2], box1[1] + box1[3]
    
    x1_2, y1_2 = box2[0], box2[1]
    x2_2, y2_2 = box2[0] + box2[2], box2[1] + box2[3]
    
    # Intersection
    x1_i = max(x1_1, x1_2)
    y1_i = max(y1_1, y1_2)
    x2_i = min(x2_1, x2_2)
    y2_i = min(y2_1, y2_2)
    
    if x2_i <= x1_i or y2_i <= y1_i:
        return 0.0
    
    intersection = (x2_i - x1_i) * (y2_i - y1_i)
    
    # Union
    area1 = box1[2] * box1[3]
    area2 = box2[2] * box2[3]
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0.0


def match_predictions_to_gt(
    pred_boxes: List[List[float]],
    pred_scores: List[float],
    gt_boxes: List[List[float]],
    iou_threshold: float = 0.5
) -> Tuple[int, int, int, List[float]]:
    """
    Match predictions to ground truth boxes.
    
    Returns:
        (true_positives, false_positives, false_negatives, matched_ious)
    """
    if len(pred_boxes) == 0:
        return 0, 0, len(gt_boxes), []
    
    if len(gt_boxes) == 0:
        return 0, len(pred_boxes), 0, []
    
    # Sort predictions by score
    sorted_indices = np.argsort(pred_scores)[::-1]
    pred_boxes = [pred_boxes[i] for i in sorted_indices]
    
    matched_gt = set()
    tp = 0
    fp = 0
    matched_ious = []
    
    for pred_box in pred_boxes:
        best_iou = 0
        best_gt_idx = -1
        
        for gt_idx, gt_box in enumerate(gt_boxes):
            if gt_idx in matched_gt:
                continue
            iou = compute_iou(pred_box, gt_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx
        
        if best_iou >= iou_threshold and best_gt_idx >= 0:
            tp += 1
            matched_gt.add(best_gt_idx)
            matched_ious.append(best_iou)
        else:
            fp += 1
    
    fn = len(gt_boxes) - len(matched_gt)
    
    return tp, fp, fn, matched_ious


def compute_ap(precisions: List[float], recalls: List[float]) -> float:
    """
    Compute Average Precision using 11-point interpolation.
    """
    if len(precisions) == 0:
        return 0.0
    
    # Add sentinel values
    precisions = [0] + list(precisions) + [0]
    recalls = [0] + list(recalls) + [1]
    
    # Compute interpolated precision
    for i in range(len(precisions) - 2, -1, -1):
        precisions[i] = max(precisions[i], precisions[i + 1])
    
    # Find points where recall changes
    ap = 0
    for i in range(1, len(recalls)):
        if recalls[i] != recalls[i - 1]:
            ap += (recalls[i] - recalls[i - 1]) * precisions[i]
    
    return ap


class MetricsTracker:
    """Track evaluation metrics across the dataset."""
    
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.all_predictions = []  # List of (score, is_tp, iou)
        self.total_gt = 0
        self.total_tp = 0
        self.total_fp = 0
        self.total_fn = 0
        self.all_ious = []
        self.per_image_metrics = []
    
    def update(self, pred_boxes, pred_scores, gt_boxes, iou_threshold=0.5):
        """Update metrics with predictions from one image."""
        tp, fp, fn, matched_ious = match_predictions_to_gt(
            pred_boxes, pred_scores, gt_boxes, iou_threshold
        )
        
        self.total_gt += len(gt_boxes)
        self.total_tp += tp
        self.total_fp += fp
        self.total_fn += fn
        self.all_ious.extend(matched_ious)
        
        # Store per-image metrics
        if len(gt_boxes) > 0:
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / len(gt_boxes)
            self.per_image_metrics.append({
                'tp': tp, 'fp': fp, 'fn': fn,
                'precision': precision, 'recall': recall,
                'avg_iou': np.mean(matched_ious) if matched_ious else 0
            })
    
    def compute_metrics(self):
        """Compute final metrics."""
        precision = self.total_tp / (self.total_tp + self.total_fp) if (self.total_tp + self.total_fp) > 0 else 0
        recall = self.total_tp / self.total_gt if self.total_gt > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        avg_iou = np.mean(self.all_ious) if self.all_ious else 0
        
        return {
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'avg_iou': avg_iou,
            'total_tp': self.total_tp,
            'total_fp': self.total_fp,
            'total_fn': self.total_fn,
            'total_gt': self.total_gt,
        }


print("Metrics implementation complete.")

## 2.5 Run Baseline Inference

In [ ]:
def run_inference(
    model: nn.Module,
    processor: DetrImageProcessor,
    dataloader: DataLoader,
    device: str,
    confidence_threshold: float = 0.5,
) -> Tuple[List[Dict], MetricsTracker, MetricsTracker]:
    """
    Run inference on the test dataset.
    
    Returns:
        predictions: List of prediction dictionaries
        metrics_50: Metrics at IoU=0.5
        metrics_75: Metrics at IoU=0.75
    """
    model.eval()
    predictions = []
    metrics_50 = MetricsTracker()
    metrics_75 = MetricsTracker()
    
    # For mAP computation across thresholds
    iou_thresholds = np.arange(0.5, 1.0, 0.05)
    metrics_per_threshold = {t: MetricsTracker() for t in iou_thresholds}
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Running inference"):
            pixel_values = batch['pixel_values'].to(device)
            orig_size = batch['orig_size']
            
            # Handle gt_boxes from DataLoader
            gt_boxes = []
            try:
                raw_gt = batch['gt_boxes']
                if isinstance(raw_gt, list):
                    # Often [tensor(x_list), tensor(y_list), tensor(w_list), tensor(h_list)]
                    if len(raw_gt) == 4 and all(len(t) == len(raw_gt[0]) for t in raw_gt):
                        extracted_boxes = []
                        num_boxes = len(raw_gt[0])
                        for i in range(num_boxes):
                            box = [
                                float(raw_gt[0][i]), 
                                float(raw_gt[1][i]), 
                                float(raw_gt[2][i]), 
                                float(raw_gt[3][i])
                            ]
                            extracted_boxes.append(box)
                        gt_boxes = extracted_boxes
                    else:
                        # Fallback for simple list of lists if collate didn't mess it up
                        gt_boxes = raw_gt
                elif isinstance(raw_gt, torch.Tensor):
                    gt_boxes = raw_gt[0].tolist()
            except Exception as e:
                # Fallback empty if parsing fails
                gt_boxes = []

            image_id = batch['image_id'][0].item() if isinstance(batch['image_id'][0], torch.Tensor) else batch['image_id'][0]
            
            # Forward pass
            outputs = model(pixel_values)
            
            # Post-process predictions
            target_sizes = torch.tensor([[orig_size[1], orig_size[0]]]).to(device)
            results = processor.post_process_object_detection(
                outputs, target_sizes=target_sizes, threshold=confidence_threshold
            )[0]
            
            # Extract predictions
            pred_boxes_xyxy = results['boxes'].cpu().numpy()
            pred_scores = results['scores'].cpu().numpy()
            pred_labels = results['labels'].cpu().numpy()
            
            # Convert to COCO format [x, y, w, h]
            pred_boxes_coco = []
            for box in pred_boxes_xyxy:
                x1, y1, x2, y2 = box
                pred_boxes_coco.append([x1, y1, x2 - x1, y2 - y1])
            
            # Filter for table class (class 0 typically)
            table_mask = pred_labels == 0
            pred_boxes_coco = [b for b, m in zip(pred_boxes_coco, table_mask) if m]
            pred_scores_filtered = pred_scores[table_mask].tolist()
            
            # Update metrics
            metrics_50.update(pred_boxes_coco, pred_scores_filtered, gt_boxes, iou_threshold=0.5)
            metrics_75.update(pred_boxes_coco, pred_scores_filtered, gt_boxes, iou_threshold=0.75)
            
            for t in iou_thresholds:
                metrics_per_threshold[t].update(pred_boxes_coco, pred_scores_filtered, gt_boxes, iou_threshold=t)
            
            # Store predictions
            predictions.append({
                'image_id': image_id,
                'pred_boxes': pred_boxes_coco,
                'pred_scores': pred_scores_filtered,
                'gt_boxes': gt_boxes,
                'file_name': batch['file_name'][0],
            })
    
    # Compute mAP@50-95
    aps = []
    for t in iou_thresholds:
        m = metrics_per_threshold[t].compute_metrics()
        aps.append(m['precision'] * m['recall'])
    
    mAP_50_95 = np.mean(aps) if aps else 0.0
    
    return predictions, metrics_50, metrics_75, mAP_50_95

# --- Run the Inference ---
print("Starting inference...")
predictions, metrics_50, metrics_75, mAP_50_95 = run_inference(
    model=td_model,
    processor=td_processor,
    dataloader=test_loader,
    device=config.DEVICE,
    confidence_threshold=0.7
)
print("Inference complete.")

## 2.6 Compute and Display Results

In [ ]:
# Compute final metrics
# Ensure variables are defined if previous cell failed
if 'metrics_50' not in locals():
    print("Metrics not found. Ensure the inference cell above ran successfully.")
    # Initialize empty trackers to avoid NameError if inference failed
    metrics_50 = MetricsTracker()
    metrics_75 = MetricsTracker()
    mAP_50_95 = 0.0
    test_dataset = [] # Avoid len() error
    predictions = []

results_50 = metrics_50.compute_metrics()
results_75 = metrics_75.compute_metrics()

print("\n" + "="*60)
print("BASELINE TD MODEL EVALUATION RESULTS")
print("="*60)
print(f"\nModel: {config.TD_MODEL_NAME}")
print(f"Test set size: {len(test_dataset)} images")
print(f"Total ground truth boxes: {results_50['total_gt']}")

print("\n" + "-"*40)
print("Metrics at IoU=0.5:")
print("-"*40)
print(f"  Precision:     {results_50['precision']:.4f}")
print(f"  Recall:        {results_50['recall']:.4f}")
print(f"  F1-Score:      {results_50['f1_score']:.4f}")
print(f"  Average IoU:   {results_50['avg_iou']:.4f}")
print(f"  mAP@50:        {results_50['precision'] * results_50['recall']:.4f}")

print("\n" + "-"*40)
print("Metrics at IoU=0.75:")
print("-"*40)
print(f"  Precision:     {results_75['precision']:.4f}")
print(f"  Recall:        {results_75['recall']:.4f}")
print(f"  F1-Score:      {results_75['f1_score']:.4f}")
print(f"  Average IoU:   {results_75['avg_iou']:.4f}")
print(f"  mAP@75:        {results_75['precision'] * results_75['recall']:.4f}")

print("\n" + "-"*40)
print("Combined Metrics:")
print("-"*40)
print(f"  mAP@50-95:     {mAP_50_95:.4f}")

print("\n" + "-"*40)
print("Detection Statistics:")
print("-"*40)
print(f"  True Positives:  {results_50['total_tp']}")
print(f"  False Positives: {results_50['total_fp']}")
print(f"  False Negatives: {results_50['total_fn']}")

In [ ]:
# Create summary dataframe
summary_data = {
    'Metric': ['Precision', 'Recall', 'F1-Score', 'Average IoU', 'mAP@50', 'mAP@75', 'mAP@50-95'],
    'Value': [
        results_50['precision'],
        results_50['recall'],
        results_50['f1_score'],
        results_50['avg_iou'],
        results_50['precision'] * results_50['recall'],
        results_75['precision'] * results_75['recall'],
        mAP_50_95
    ]
}

summary_df = pd.DataFrame(summary_data)
summary_df['Value'] = summary_df['Value'].apply(lambda x: f"{x:.4f}")

print("\nSummary Table:")
print(summary_df.to_string(index=False))

# Save results
baseline_results = {
    'model': config.TD_MODEL_NAME,
    'task': 'Table Detection (TD)',
    'metrics_iou_50': results_50,
    'metrics_iou_75': results_75,
    'mAP_50_95': mAP_50_95,
    'num_test_images': len(test_dataset),
}

results_path = os.path.join(config.OUTPUT_DIR, 'td_baseline_results.json')
with open(results_path, 'w') as f:
    json.dump(baseline_results, f, indent=2, default=str)

print(f"\nResults saved to {results_path}")

## 2.7 Visualize Sample Detections

In [ ]:
def visualize_detection(
    image_path: str,
    pred_boxes: List[List[float]],
    pred_scores: List[float],
    gt_boxes: List[List[float]],
    figsize: Tuple[int, int] = (14, 10)
):
    """
    Visualize predictions vs ground truth.
    
    Args:
        image_path: Path to the image
        pred_boxes: Predicted boxes in COCO format [x, y, w, h]
        pred_scores: Confidence scores
        gt_boxes: Ground truth boxes in COCO format
        figsize: Figure size
    """
    image = Image.open(image_path)
    
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # Left: Ground Truth
    axes[0].imshow(image)
    axes[0].set_title('Ground Truth', fontsize=14)
    for box in gt_boxes:
        x, y, w, h = box
        rect = patches.Rectangle(
            (x, y), w, h,
            linewidth=3, edgecolor='green', facecolor='none'
        )
        axes[0].add_patch(rect)
    axes[0].axis('off')
    
    # Right: Predictions
    axes[1].imshow(image)
    axes[1].set_title('Predictions (Baseline TD)', fontsize=14)
    for box, score in zip(pred_boxes, pred_scores):
        x, y, w, h = box
        rect = patches.Rectangle(
            (x, y), w, h,
            linewidth=3, edgecolor='red', facecolor='none'
        )
        axes[1].add_patch(rect)
        axes[1].text(x, y - 5, f'{score:.2f}', fontsize=10, color='red',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()


# Visualize a few sample predictions
print("\nSample Detections:")
print("="*60)

# Select samples with different characteristics
sample_indices = [0, len(predictions)//4, len(predictions)//2, -1]

for idx in sample_indices:
    if idx < len(predictions):
        pred = predictions[idx]
        image_path = os.path.join(config.IMAGES_DIR, pred['file_name'])
        
        if os.path.exists(image_path):
            print(f"\nImage: {pred['file_name']}")
            print(f"  GT boxes: {len(pred['gt_boxes'])}")
            print(f"  Predicted boxes: {len(pred['pred_boxes'])}")
            
            visualize_detection(
                image_path=image_path,
                pred_boxes=pred['pred_boxes'],
                pred_scores=pred['pred_scores'],
                gt_boxes=pred['gt_boxes']
            )

## 2.8 Metrics Distribution Visualization

In [ ]:
# Plot IoU distribution
if metrics_50.all_ious:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # IoU histogram
    axes[0].hist(metrics_50.all_ious, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[0].axvline(x=np.mean(metrics_50.all_ious), color='red', linestyle='--', 
                   label=f'Mean: {np.mean(metrics_50.all_ious):.3f}')
    axes[0].axvline(x=0.5, color='green', linestyle='--', label='IoU=0.5 threshold')
    axes[0].set_xlabel('IoU Score', fontsize=12)
    axes[0].set_ylabel('Frequency', fontsize=12)
    axes[0].set_title('IoU Distribution of Matched Predictions', fontsize=14)
    axes[0].legend()
    
    # Per-image metrics
    if metrics_50.per_image_metrics:
        per_image_df = pd.DataFrame(metrics_50.per_image_metrics)
        axes[1].scatter(per_image_df['recall'], per_image_df['precision'], 
                       c=per_image_df['avg_iou'], cmap='viridis', alpha=0.6)
        axes[1].set_xlabel('Recall', fontsize=12)
        axes[1].set_ylabel('Precision', fontsize=12)
        axes[1].set_title('Per-Image Precision vs Recall', fontsize=14)
        cbar = plt.colorbar(axes[1].collections[0], ax=axes[1])
        cbar.set_label('Average IoU')
    
    plt.tight_layout()
    plt.savefig(os.path.join(config.OUTPUT_DIR, 'td_baseline_metrics.png'), dpi=150)
    plt.show()
else:
    print("No matched predictions to visualize.")

In [ ]:
# Summary bar chart
fig, ax = plt.subplots(figsize=(10, 6))

metrics_names = ['Precision', 'Recall', 'F1-Score', 'Avg IoU', 'mAP@50', 'mAP@75']
metrics_values = [
    results_50['precision'],
    results_50['recall'],
    results_50['f1_score'],
    results_50['avg_iou'],
    results_50['precision'] * results_50['recall'],
    results_75['precision'] * results_75['recall'],
]

colors = plt.cm.Blues(np.linspace(0.4, 0.8, len(metrics_names)))
bars = ax.bar(metrics_names, metrics_values, color=colors, edgecolor='navy', linewidth=1.2)

# Add value labels
for bar, val in zip(bars, metrics_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylim(0, 1.1)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Baseline TD Model Performance Metrics', fontsize=14, fontweight='bold')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='50% threshold')

plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_DIR, 'td_baseline_summary.png'), dpi=150)
plt.show()

print("\n" + "="*60)
print("Part 2 Complete - TD Baseline Inference Done")
print("="*60)
print("\nNext: Run Part 3 for TSR Enhanced Architecture & LoRA Fine-tuning")